# Django REST Framework: Complete Structured Guide

---
## Module 1: Introduction to APIs
---

### 1.1 What is an API?
* API = Application Programming Interface
* Allows communication between applications

Example:
```
React Frontend
        ↓
REST API
        ↓
Django Backend
        ↓
PostgreSQL
```

### 1.2 Why APIs?

**Traditional Web Application**
```
Browser
  ↓
Django
  ↓
HTML
```
**Modern Application**
```
Web App
Mobile App
Third Party System
      ↓
 REST API
      ↓
 Backend
```

**Benefits:**

* Platform independent
* Reusable
* Scalable

### 1.3 What is REST?
REST = Representational State Transfer

**Principles:**

* Client Server
* Stateless
* Uniform Interface
* Resource Based URLs

**Example:**

```
/users
/products
/orders
```

### 1.4 HTTP Methods
| Method | Purpose        |
| ------ | -------------- |
| GET    | Fetch          |
| POST   | Create         |
| PUT    | Update         |
| PATCH  | Partial Update |
| DELETE | Delete         |

**Example:**
```
GET     /students
POST    /students
GET     /students/1
PUT     /students/1
DELETE  /students/1
```

### 1.5 HTTP Status Codes
| Code | Meaning               |
| ---- | --------------------- |
| 200  | Success               |
| 201  | Created               |
| 400  | Bad Request           |
| 401  | Unauthorized          |
| 403  | Forbidden             |
| 404  | Not Found             |
| 500  | Internal Server Error |


---
## Module 2: Introduction to DRF
---

### 2.1 What is Django REST Framework?
A toolkit built on top of Django for building REST APIs.

Features:

* Serialization
* Authentication
* Authorization
* Validation
* Pagination
* Filtering
* Browsable API

### 2.2 DRF Architecture

```
Request
   ↓
URL
   ↓
View
   ↓
Serializer
   ↓
Model
   ↓
Database

Database
   ↓
Model
   ↓
Serializer
   ↓
Response
```
This is the most important DRF diagram.

### 2.3 Installation & Project Setup

**Install the packages:**
```bash
pip install djangorestframework
pip install djangorestframework-simplejwt   # for JWT auth
pip install django-filter                   # for filtering
```

**Register DRF in `settings.py`:**
```python
INSTALLED_APPS = [
    # ... default Django apps ...
    "rest_framework",
    "students",   # your app
]
```

**Wire up the URLs in `project/urls.py`:**
```python
from django.urls import path, include

urlpatterns = [
    path("api/", include("students.urls")),
]
```

> **Note for this notebook:** instead of a full project, the next cell configures Django *in-memory* using `settings.configure()` so that every example below actually runs. Run the cells in order from top to bottom.

In [1]:
# One-time Django + DRF setup so every example in this notebook is runnable.
# In a real project these settings live in settings.py and Django loads them for you.
import os

# Jupyter runs cells inside an asyncio event loop; this lets Django's ORM run
# synchronously inside the notebook. Not needed in a normal Django project.
os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "1")

import django
from django.conf import settings

if not settings.configured:
    settings.configure(
        DEBUG=True,
        SECRET_KEY="training-only-secret-key",
        ALLOWED_HOSTS=["*"],
        DATABASES={
            "default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": ":memory:",
            }
        },
        INSTALLED_APPS=[
            "django.contrib.contenttypes",
            "django.contrib.auth",
            "rest_framework",
        ],
        REST_FRAMEWORK={
            "DEFAULT_PAGINATION_CLASS": "rest_framework.pagination.PageNumberPagination",
            "PAGE_SIZE": 2,
        },
    )
    django.setup()

import rest_framework

print("Django version:", django.get_version())
print("DRF version   :", rest_framework.VERSION)
print("Settings configured:", settings.configured)

Django version: 5.2
DRF version   : 3.14.0
Settings configured: True


---
## Module 3: Sample Project Walkthrough
---

### 3.1 Create a Student API

We will build:

```Student Management API```

**Operations:**

* Create Student
* Get Student
* Update Student
* Delete Student

### 3.2 Student Model

```python
# students/models.py
from django.db import models


class Student(models.Model):
    name = models.CharField(max_length=100)
    age = models.IntegerField()
    email = models.EmailField()

    def __str__(self):
        return self.name
```

Database Table:
| id | name | age |
| -- | ---- | --- |
| 1  | John | 22  |


In [2]:
# Define the model and create its table in the in-memory database.
from django.db import models, connection


class Student(models.Model):
    name = models.CharField(max_length=100)
    age = models.IntegerField()
    email = models.EmailField()

    class Meta:
        app_label = "demo"  # required because we're not inside a real app package

    def __str__(self):
        return self.name


# Normally `python manage.py migrate` builds tables; here we create it directly.
with connection.schema_editor() as schema_editor:
    schema_editor.create_model(Student)

print("Student table created:", Student._meta.db_table)

Student table created: demo_student


---
## Module 4: Serializers
---

### 4.1 What is Serialization?
Converts:
Python Object → JSON

JSON → Python Object

Example:

```
student.name
```
becomes
```
{
  "name": "John"
}
```

### 4.2 Serializer Example
```python
# students/serializers.py
from rest_framework import serializers
from .models import Student


class StudentSerializer(serializers.ModelSerializer):
    class Meta:
        model = Student
        fields = "__all__"
```

In [3]:
from rest_framework import serializers


class StudentSerializer(serializers.ModelSerializer):
    class Meta:
        model = Student
        fields = "__all__"


# Deserialization: incoming JSON -> validated object saved to the DB
serializer = StudentSerializer(data={"name": "John", "age": 22, "email": "john@example.com"})
print("is_valid  :", serializer.is_valid())
student = serializer.save()
print("created id:", student.id)

# Serialization: model instance -> JSON-ready dict
print("serialized:", dict(StudentSerializer(student).data))

is_valid  : True
created id: 1
serialized: {'id': 1, 'name': 'John', 'age': 22, 'email': 'john@example.com'}


### 4.3 Why Serializer?
Responsibilities:

* Validation
* Serialization
* Deserialization

```python
serializer.is_valid()
serializer.save()
```

In [4]:
# Field-level validation: reject students under 18 using validate_<field>.
class StudentSerializer(serializers.ModelSerializer):
    class Meta:
        model = Student
        fields = "__all__"

    def validate_age(self, value):
        if value < 18:
            raise serializers.ValidationError("Age must be 18 or above.")
        return value


valid = StudentSerializer(data={"name": "Mary", "age": 25, "email": "mary@example.com"})
print("age 25 -> is_valid:", valid.is_valid())

invalid = StudentSerializer(data={"name": "Kid", "age": 10, "email": "kid@example.com"})
print("age 10 -> is_valid:", invalid.is_valid())
print("errors          :", invalid.errors)

age 25 -> is_valid: True
age 10 -> is_valid: False
errors          : {'age': [ErrorDetail(string='Age must be 18 or above.', code='invalid')]}


---
## Module 5: Views
---

### 5.1 Function-Based Views (`@api_view`)

A **function-based view (FBV)** is a normal Python function decorated with
`@api_view([...])` listing the HTTP methods it handles. You check
`request.method` to decide what to do.

```python
# students/views.py
from rest_framework.decorators import api_view
from rest_framework.response import Response
from rest_framework import status


# You choose the function name (e.g. student_list_create, list_students, ...).
@api_view(["GET", "POST"])
def student_list_create(request):
    if request.method == "GET":
        students = Student.objects.all()
        return Response(StudentSerializer(students, many=True).data)

    serializer = StudentSerializer(data=request.data)
    serializer.is_valid(raise_exception=True)
    serializer.save()
    return Response(serializer.data, status=status.HTTP_201_CREATED)
```

**URL wiring (map the URL straight to the function):**
```python
# students/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path("students/", views.student_list_create),
]
```

> **Why our organization uses function-based views:** with FBVs you control the
> function name (`student_list_create`, `create_student`, ...). With class-based
> views the handler methods **must** be named `get`, `post`, `put`, `patch`,
> `delete` — you cannot rename them. Because we standardize on descriptive,
> custom function names, **we use FBVs throughout** and treat the class-based
> sections that follow (APIView, generics, ViewSets) as reference only.

In [5]:
# A runnable function-based view. APIRequestFactory simulates HTTP requests
# without starting a real web server.
from rest_framework.decorators import api_view
from rest_framework.response import Response
from rest_framework import status
from rest_framework.test import APIRequestFactory

factory = APIRequestFactory()


# The function name is ours to choose - here: student_list_create.
@api_view(["GET", "POST"])
def student_list_create(request):
    if request.method == "GET":
        students = Student.objects.all()
        return Response(StudentSerializer(students, many=True).data)

    serializer = StudentSerializer(data=request.data)
    serializer.is_valid(raise_exception=True)
    serializer.save()
    return Response(serializer.data, status=status.HTTP_201_CREATED)


# Note: a function-based view is called directly (no `.as_view()`).
post_resp = student_list_create(
    factory.post("/students/", {"name": "Neha", "age": 28, "email": "neha@example.com"}, format="json")
)
print("POST ->", post_resp.status_code, dict(post_resp.data))

get_resp = student_list_create(factory.get("/students/"))
print("GET  ->", get_resp.status_code, "| students returned:", len(get_resp.data))

POST -> 201 {'id': 2, 'name': 'Neha', 'age': 28, 'email': 'neha@example.com'}
GET  -> 200 | students returned: 2


### 5.2 APIView (class-based)
Coming from FBVs, think of `APIView` as the **same view written as a class**:
instead of one function that branches on `request.method`, each HTTP verb becomes
its own method named `get`, `post`, `put`, `patch`, or `delete` (names are fixed).
```python
from rest_framework.views import APIView
from rest_framework.response import Response


class StudentView(APIView):

    def get(self, request):     # was: if request.method == "GET"
        ...

    def post(self, request):    # was: if request.method == "POST"
        ...
```

| Function-based (`@api_view`) | APIView (class-based) |
| --- | --- |
| One function, branch on `request.method` | One method per HTTP verb |
| Any function name | Methods must be `get` / `post` / ... |
| `path("students/", view_func)` | `path("students/", StudentView.as_view())` |

### 5.3 Request Lifecycle
```
Client
 ↓
URL
 ↓
APIView
 ↓
Serializer
 ↓
Database
 ↓
Response
```

### 5.4 GET Example
```python
students = Student.objects.all()

serializer = StudentSerializer(students, many=True)

return Response(serializer.data)
```


### 5.5 POST Example
```python
serializer = StudentSerializer(data=request.data)

if serializer.is_valid():
    serializer.save()
    return Response(serializer.data, status=201)
return Response(serializer.errors, status=400)
```

In [6]:
# A runnable APIView. We use APIRequestFactory to simulate HTTP requests
# without starting a real web server.
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from rest_framework.test import APIRequestFactory


class StudentListCreate(APIView):
    def get(self, request):
        students = Student.objects.all()
        return Response(StudentSerializer(students, many=True).data)

    def post(self, request):
        serializer = StudentSerializer(data=request.data)
        serializer.is_valid(raise_exception=True)
        serializer.save()
        return Response(serializer.data, status=status.HTTP_201_CREATED)


factory = APIRequestFactory()

post_req = factory.post(
    "/students/", {"name": "Ravi", "age": 30, "email": "ravi@example.com"}, format="json"
)
post_resp = StudentListCreate.as_view()(post_req)
print("POST ->", post_resp.status_code, dict(post_resp.data))

get_resp = StudentListCreate.as_view()(factory.get("/students/"))
print("GET  ->", get_resp.status_code, "| students returned:", len(get_resp.data))

POST -> 201 {'id': 3, 'name': 'Ravi', 'age': 30, 'email': 'ravi@example.com'}
GET  -> 200 | students returned: 3


---
## Module 6: Generic Views
---

### 6.1 Why Generic Views?
With FBVs (and APIView) you write the same CRUD steps again and again: query the
model, serialize, validate, save, return a response. **Generic views** are
pre-built classes that already contain that logic, so you only declare
`queryset` and `serializer_class`.

> **Coming from FBVs:** a generic view is *"an `@api_view` function whose body DRF
> already wrote for you."* We still write plain functions in our projects, but
> generics appear in a lot of DRF code, so it helps to recognize them.

### 6.2 Available Generic Views
Each class handles the CRUD action(s) you would otherwise hand-write in an FBV.
Pick the one whose verbs match the endpoint you need:
```python
ListAPIView                   # GET    list      -> Student.objects.all()
CreateAPIView                 # POST   create
RetrieveAPIView               # GET    one (by pk)
UpdateAPIView                 # PUT/PATCH update
DestroyAPIView                # DELETE remove
ListCreateAPIView             # GET list + POST create      (collection endpoint)
RetrieveUpdateDestroyAPIView  # GET + PUT/PATCH + DELETE     (detail endpoint)
```
Two endpoints (`ListCreateAPIView` + `RetrieveUpdateDestroyAPIView`) together
give a full CRUD API.

### 6.3 Example: FBV vs Generic View
The function-based version you write by hand:
```python
@api_view(["GET", "POST"])
def student_list_create(request):
    if request.method == "GET":
        students = Student.objects.all()
        return Response(StudentSerializer(students, many=True).data)
    serializer = StudentSerializer(data=request.data)
    serializer.is_valid(raise_exception=True)
    serializer.save()
    return Response(serializer.data, status=201)
```
The exact same endpoint as a generic view — DRF supplies the body:
```python
from rest_framework.generics import ListCreateAPIView


class StudentView(ListCreateAPIView):
    queryset = Student.objects.all()
    serializer_class = StudentSerializer
```
Both expose `GET /students/` and `POST /students/`. The generic version is
shorter; the FBV version is explicit. *(The runnable cell below uses the generic view.)*

In [7]:
from rest_framework.generics import ListCreateAPIView


class StudentGenericView(ListCreateAPIView):
    queryset = Student.objects.all().order_by("id")
    serializer_class = StudentSerializer


# Same GET endpoint as the APIView above, but with almost no code.
resp = StudentGenericView.as_view()(factory.get("/students/"))
print("Generic view status     :", resp.status_code)
print("Paginated response keys :", list(resp.data.keys()))
print("Total count             :", resp.data["count"])
print("Items on this page      :", len(resp.data["results"]))

Generic view status     : 200
Paginated response keys : ['count', 'next', 'previous', 'results']
Total count             : 3
Items on this page      : 2


---
## Module 7: ViewSets & Routers
---

### 7.1 ViewSet
A `ViewSet` bundles **all CRUD actions for one resource into a single class**.
A `ModelViewSet` gives the full set from just two attributes:
```python
from rest_framework.viewsets import ModelViewSet


class StudentViewSet(ModelViewSet):
    queryset = Student.objects.all()
    serializer_class = StudentSerializer
```
Actions provided (names are fixed): `list`, `create`, `retrieve`, `update`,
`partial_update`, `destroy` — covering GET, POST, PUT, PATCH, DELETE.

> **Coming from FBVs:** one `ModelViewSet` replaces the **two functions**
> (`students_view` + `student_detail_view`) you'd write for full CRUD. The
> function-based equivalent is shown in Section 7.3 below.

### 7.2 Router
A router auto-generates the URL patterns for a ViewSet, so you don't write
`path()` lines yourself:
```python
# students/urls.py
from rest_framework.routers import DefaultRouter
from .views import StudentViewSet

router = DefaultRouter()
router.register("students", StudentViewSet)

urlpatterns = router.urls
```
Generated endpoints:
```
GET, POST              /students/      -> list, create
GET, PUT, PATCH, DELETE /students/1/   -> retrieve, update, partial_update, destroy
```
> **Coming from FBVs:** the router replaces the explicit `path("students/", ...)`
> and `path("students/<int:pk>/", ...)` lines you'd write for function views.

In [8]:
from rest_framework.viewsets import ModelViewSet
from rest_framework.routers import DefaultRouter


class StudentViewSet(ModelViewSet):
    queryset = Student.objects.all().order_by("id")
    serializer_class = StudentSerializer


router = DefaultRouter()
router.register("students", StudentViewSet)

print("URLs generated automatically by the router:")
for route in router.urls:
    print("  ", route.pattern)

# A ViewSet maps actions to HTTP methods. Here we invoke the 'list' action.
list_view = StudentViewSet.as_view({"get": "list"})
resp = list_view(factory.get("/students/"))
print("\nViewSet list -> status", resp.status_code, "| count:", resp.data["count"])

URLs generated automatically by the router:
   ^students/$
   ^students\.(?P<format>[a-z0-9]+)/?$
   ^students/(?P<pk>[^/.]+)/$
   ^students/(?P<pk>[^/.]+)\.(?P<format>[a-z0-9]+)/?$
   ^$
   ^\.(?P<format>[a-z0-9]+)/?$

ViewSet list -> status 200 | count: 3


### 7.3 The Same Full CRUD with Function-Based Views
For comparison, here is the identical full CRUD using **function-based views** —
two functions (names of your choice), wired with explicit `path()` entries.
```python
# students/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path("students/", views.students_view),                 # GET list, POST create
    path("students/<int:pk>/", views.student_detail_view),  # GET/PUT/PATCH/DELETE one
]
```
The next cell runs this full CRUD cycle live.

In [9]:
# The same full CRUD as the ViewSet above, written with FUNCTION-BASED views.
from rest_framework.decorators import api_view
from rest_framework.response import Response
from rest_framework import status
from rest_framework.test import APIRequestFactory
from django.shortcuts import get_object_or_404

fbv_factory = APIRequestFactory()


# Collection endpoint: list all students + create a new one.
@api_view(["GET", "POST"])
def students_view(request):
    if request.method == "GET":
        students = Student.objects.all().order_by("id")
        return Response(StudentSerializer(students, many=True).data)

    serializer = StudentSerializer(data=request.data)
    serializer.is_valid(raise_exception=True)
    serializer.save()
    return Response(serializer.data, status=status.HTTP_201_CREATED)


# Detail endpoint: retrieve / update / delete a single student.
@api_view(["GET", "PUT", "PATCH", "DELETE"])
def student_detail_view(request, pk):
    student = get_object_or_404(Student, pk=pk)

    if request.method == "GET":
        return Response(StudentSerializer(student).data)

    if request.method in ("PUT", "PATCH"):
        partial = request.method == "PATCH"
        serializer = StudentSerializer(student, data=request.data, partial=partial)
        serializer.is_valid(raise_exception=True)
        serializer.save()
        return Response(serializer.data)

    student.delete()
    return Response(status=status.HTTP_204_NO_CONTENT)


# --- Drive the full CRUD cycle (same operations as the ViewSet cell) ---
r = students_view(fbv_factory.post("/students/", {"name": "Vikram", "age": 26, "email": "vikram@example.com"}, format="json"))
new_id = r.data["id"]
print("CREATE ->", r.status_code, dict(r.data))

r = student_detail_view(fbv_factory.get(f"/students/{new_id}/"), pk=new_id)
print("READ   ->", r.status_code, dict(r.data))

r = student_detail_view(fbv_factory.patch(f"/students/{new_id}/", {"age": 27}, format="json"), pk=new_id)
print("UPDATE ->", r.status_code, dict(r.data))

r = student_detail_view(fbv_factory.delete(f"/students/{new_id}/"), pk=new_id)
print("DELETE ->", r.status_code)
print("Student still exists after delete?", Student.objects.filter(id=new_id).exists())

CREATE -> 201 {'id': 4, 'name': 'Vikram', 'age': 26, 'email': 'vikram@example.com'}
READ   -> 200 {'id': 4, 'name': 'Vikram', 'age': 26, 'email': 'vikram@example.com'}
UPDATE -> 200 {'id': 4, 'name': 'Vikram', 'age': 27, 'email': 'vikram@example.com'}
DELETE -> 204
Student still exists after delete? False


---
## Module 8: Authentication & Security
---

### 8.1 Authentication
Question: **Who are you?**

Types:

* Session Authentication
* Token Authentication
* JWT Authentication

### 8.2 JWT Authentication
Flow:
```
Login
 ↓
JWT Token
 ↓
Authorization Header
 ↓
API Access
```
Request header sent by the client:
```
Authorization: Bearer eyJ...
```

**Setup with `djangorestframework-simplejwt`:**
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": (
        "rest_framework_simplejwt.authentication.JWTAuthentication",
    ),
}
```
```python
# urls.py
from rest_framework_simplejwt.views import (
    TokenObtainPairView,
    TokenRefreshView,
)

urlpatterns += [
    path("api/token/", TokenObtainPairView.as_view()),
    path("api/token/refresh/", TokenRefreshView.as_view()),
]
```

### 8.3 Permissions
Question: **What are you allowed to do?**
```python
from rest_framework.permissions import IsAuthenticated


class StudentViewSet(ModelViewSet):
    queryset = Student.objects.all()
    serializer_class = StudentSerializer
    permission_classes = [IsAuthenticated]
```
Common built-in permission classes:
```python
AllowAny           # anyone
IsAuthenticated    # logged-in users only
IsAdminUser        # staff users only
IsAuthenticatedOrReadOnly  # read for all, write for logged-in
```

### 8.4 Serializer Validation
Field-level validation lives on the serializer as `validate_<field>`:
```python
from rest_framework import serializers


class StudentSerializer(serializers.ModelSerializer):
    class Meta:
        model = Student
        fields = "__all__"

    def validate_age(self, value):
        if value < 18:
            raise serializers.ValidationError("Age must be 18 or above.")
        return value
```

---
## Module 9: Production Features
---

### 9.1 Pagination
Problem:
```
1 Million Records
```
Solution: split results into pages.
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_PAGINATION_CLASS": "rest_framework.pagination.PageNumberPagination",
    "PAGE_SIZE": 10,
}
```
Benefits:

* Faster response
* Lower memory usage

In [10]:
# PAGE_SIZE is 2 in our settings, so each page returns at most 2 records.
# First make sure we have several students to page through.
for i in range(1, 6):
    Student.objects.get_or_create(
        email=f"bulk{i}@example.com",
        defaults={"name": f"Bulk Student {i}", "age": 20 + i},
    )

page1 = StudentGenericView.as_view()(factory.get("/students/"))
page2 = StudentGenericView.as_view()(factory.get("/students/?page=2"))

print("Total students:", page1.data["count"])
print("Page 1 items :", [s["name"] for s in page1.data["results"]])
print("Page 2 items :", [s["name"] for s in page2.data["results"]])
print("Has next page:", page1.data["next"] is not None)

Total students: 8
Page 1 items : ['John', 'Neha']
Page 2 items : ['Ravi', 'Bulk Student 1']
Has next page: True


### 9.2 Filtering
```
/students?name=john
/students?age=25
```
Enable with `django-filter`:
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_FILTER_BACKENDS": ["django_filters.rest_framework.DjangoFilterBackend"],
}
```
```python
# views.py
class StudentViewSet(ModelViewSet):
    queryset = Student.objects.all()
    serializer_class = StudentSerializer
    filterset_fields = ["name", "age"]
```

### 9.3 Search
```
/students?search=john
```
```python
from rest_framework.filters import SearchFilter


class StudentViewSet(ModelViewSet):
    queryset = Student.objects.all()
    serializer_class = StudentSerializer
    filter_backends = [SearchFilter]
    search_fields = ["name", "email"]
```
Useful for large datasets.

### 9.4 Throttling
Throttling limits how many requests a client can make in a time window (rate limiting).
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_THROTTLE_CLASSES": [
        "rest_framework.throttling.AnonRateThrottle",
        "rest_framework.throttling.UserRateThrottle",
    ],
    "DEFAULT_THROTTLE_RATES": {
        "anon": "100/day",   # anonymous users
        "user": "1000/day",  # authenticated users
    },
}
```

### 9.5 API Versioning
Versioning lets you evolve an API without breaking existing clients.
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_VERSIONING_CLASS": "rest_framework.versioning.URLPathVersioning",
}
```
```
GET /api/v1/students/
GET /api/v2/students/
```

### 9.6 Nested / Related Serializers
When a model has relationships, a serializer can embed related data instead of just IDs.
```python
class CourseSerializer(serializers.ModelSerializer):
    class Meta:
        model = Course
        fields = ["id", "title"]


class StudentSerializer(serializers.ModelSerializer):
    courses = CourseSerializer(many=True, read_only=True)  # nested output

    class Meta:
        model = Student
        fields = ["id", "name", "age", "email", "courses"]
```

---
## Module 10: Error Handling
---

### 10.1 Exception Handling
```python
from rest_framework.response import Response
from rest_framework import status


try:
    student = Student.objects.get(id=pk)
except Student.DoesNotExist:
    return Response(
        {"error": "Student not found"},
        status=status.HTTP_404_NOT_FOUND,
    )
```
DRF automatically converts many common exceptions (validation errors, `Http404`, permission denied) into proper JSON responses with the right status codes.

---
## Module 11: Testing DRF APIs
---

DRF provides `APITestCase` and `APIClient` for writing automated tests.

```python
# students/tests.py
from rest_framework.test import APITestCase
from rest_framework import status


class StudentAPITests(APITestCase):
    def test_create_student(self):
        payload = {"name": "Test", "age": 21, "email": "test@example.com"}
        response = self.client.post("/api/students/", payload, format="json")
        self.assertEqual(response.status_code, status.HTTP_201_CREATED)

    def test_list_students(self):
        response = self.client.get("/api/students/")
        self.assertEqual(response.status_code, status.HTTP_200_OK)
```

Run the tests with:
```bash
python manage.py test
```

In [11]:
# A runnable mini-test (in a real project you'd use APITestCase + `manage.py test`).
test_factory = APIRequestFactory()

# Test 1: creating a valid student returns 201 CREATED
resp = StudentListCreate.as_view()(
    test_factory.post(
        "/students/",
        {"name": "Test User", "age": 21, "email": "testuser@example.com"},
        format="json",
    )
)
assert resp.status_code == 201, resp.data
print("test_create_student PASSED -> status", resp.status_code)

# Test 2: listing students returns 200 OK
resp = StudentListCreate.as_view()(test_factory.get("/students/"))
assert resp.status_code == 200
print("test_list_students   PASSED -> status", resp.status_code)

test_create_student PASSED -> status 201
test_list_students   PASSED -> status 200


---
## Module 12: Interview Questions (with Answers)
---

**1. What is DRF?**
A toolkit built on top of Django for building REST APIs. It adds serialization, authentication, viewsets, routers, pagination, and a browsable API.

**2. What is a Serializer?**
A component that converts complex types (model instances, querysets) to JSON and back, and also validates incoming data.

**3. Difference between APIView and ViewSet?**
`APIView` is low-level; you write each HTTP method by hand. A `ViewSet` groups related actions (list/create/retrieve/update/destroy) into one class and works with a router to auto-generate URLs.

**4. Difference between PUT and PATCH?**
`PUT` replaces the whole resource (all fields required). `PATCH` updates only the fields provided (partial update).

**5. What is JWT?**
JSON Web Token: a signed, stateless token sent in the `Authorization: Bearer <token>` header to authenticate API requests without server-side sessions.

**6. What is Authentication?**
Verifying *who* the user is (identity).

**7. What is Authorization?**
Determining *what* an authenticated user is allowed to do (permissions).

**8. What is Pagination?**
Splitting large result sets into smaller pages to improve response time and reduce memory usage.

**9. What is ModelViewSet?**
A viewset that provides full CRUD (`list`, `create`, `retrieve`, `update`, `partial_update`, `destroy`) from just a `queryset` and `serializer_class`.

**10. What is a Router?**
An object (e.g. `DefaultRouter`) that automatically generates URL patterns for a viewset, so you don't write them manually.

**11. Difference between function-based views (FBV) and class-based views (CBV)?**
An FBV is a single function decorated with `@api_view([...])`; you choose the function name and handle each HTTP method with `if request.method == ...`. A CBV (`APIView`, generics, `ViewSet`) is a class whose handler methods must be named `get`, `post`, `put`, `patch`, `delete`. FBVs give full control and naming freedom; CBVs reduce boilerplate but lock method names. *(Our organization standardizes on FBVs.)*

---
## Module 13: Complete CRUD API Project
---

We tie everything together into one working CRUD API.

```
Student Model
      ↓
Serializer
      ↓
ViewSet
      ↓
Router
      ↓
CRUD Endpoints
```

**In a real project the files are split like this:**
```
students/
  models.py        # Student model
  serializers.py   # StudentSerializer
  views.py         # StudentViewSet
  urls.py          # router.register("students", StudentViewSet)
project/
  settings.py      # INSTALLED_APPS += ["rest_framework", "students"]
  urls.py          # path("api/", include("students.urls"))
```

The next code cell runs the **entire flow live** inside this notebook (in-memory database) so you can see the actual CRUD requests and JSON responses.

This capstone uses the class-based `ModelViewSet`. The **function-based** version
of the same full CRUD was shown alongside it in **Section 7.3**.

In [12]:
# Complete CRUD walkthrough using the ViewSet, driven by APIRequestFactory.
crud = APIRequestFactory()

# CREATE
create = StudentViewSet.as_view({"post": "create"})
r = create(crud.post("/students/", {"name": "Asha", "age": 23, "email": "asha@example.com"}, format="json"))
new_id = r.data["id"]
print("CREATE ->", r.status_code, dict(r.data))

# READ (retrieve one)
retrieve = StudentViewSet.as_view({"get": "retrieve"})
r = retrieve(crud.get(f"/students/{new_id}/"), pk=new_id)
print("READ   ->", r.status_code, dict(r.data))

# UPDATE (PATCH = partial update)
partial = StudentViewSet.as_view({"patch": "partial_update"})
r = partial(crud.patch(f"/students/{new_id}/", {"age": 24}, format="json"), pk=new_id)
print("UPDATE ->", r.status_code, dict(r.data))

# DELETE
destroy = StudentViewSet.as_view({"delete": "destroy"})
r = destroy(crud.delete(f"/students/{new_id}/"), pk=new_id)
print("DELETE ->", r.status_code)

# Confirm deletion
print("Student still exists after delete?", Student.objects.filter(id=new_id).exists())

CREATE -> 201 {'id': 11, 'name': 'Asha', 'age': 23, 'email': 'asha@example.com'}
READ   -> 200 {'id': 11, 'name': 'Asha', 'age': 23, 'email': 'asha@example.com'}
UPDATE -> 200 {'id': 11, 'name': 'Asha', 'age': 24, 'email': 'asha@example.com'}
DELETE -> 204
Student still exists after delete? False
